In [1]:
pip install transformers torch datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 6.1 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-m

In [2]:
import json

# Load the SQuAD dataset
with open('/content/final_squad_dataset.json', 'r') as file:
    dataset = json.load(file)

# Display the first few entries
print(dataset)

[{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represented the AFC at Super Bowl 50?', 'answers': {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'], 'an

In [3]:
import random

def create_structured_prompt(dataset, test_example, num_demonstrations=3):
    # Filter dataset to find samples with the same context
    same_context_samples = [sample for sample in dataset if sample['context'] == test_example['context'] and sample['id'] != test_example['id']]

    if len(same_context_samples) <= num_demonstrations:
        raise ValueError("Not enough unique samples with the same context to select the required number of demonstrations.")

    demonstrations = random.sample(same_context_samples, num_demonstrations)

    f = "Give me an answer for the following question by selecting one of the 3 possible answers 1, 2, 3 based on the provided context.\n"

    prompt = ''
    for demo in demonstrations:
        prompt += f
        prompt += f"Context: \"{demo['context']}\"\n"
        prompt += f"Question: \"{demo['question']}\"\n"

        remaining_samples = [s for s in same_context_samples if s != demo]
        possible_answers = random.sample(remaining_samples, 2)
        possible_answers.append(demo)

        random.shuffle(possible_answers)

        options = ['1', '2', '3']
        correct_option = None
        prompt += "Possible Answers:\n"
        for idx, sample in enumerate(possible_answers):
            answer_text = sample['answers']['text'][0]
            option = options[idx]
            prompt += f"{option}: {answer_text}\n"
            if sample == demo:
                correct_option = option

        #correct_option = random.choice(options)
        prompt += f"Correct answer: {correct_option}\n"
        prompt += f"Reasoning: \"{demo['reasoning']}\"\n"


    prompt += f
    prompt += f"Context: \"{test_example['context']}\"\n"
    prompt += f"Question: \"{test_example['question']}\"\n"

    remaining_samples = [s for s in same_context_samples if s != test_example]
    if len(remaining_samples) >= 2:
        possible_answers = random.sample(remaining_samples, 2)
    else:
        possible_answers = remaining_samples[:2]  # Handle case where not enough samples are available
    possible_answers.append(test_example)

    random.shuffle(possible_answers)

    options = ['1', '2', '3']
    correct_option = None
    prompt += "Possible Answers:\n"
    for idx, sample in enumerate(possible_answers):
        answer_text = sample['answers']['text'][0]
        option = options[idx]
        prompt += f"{option}: {answer_text}\n"
        if sample == test_example:
                correct_option = option

    prompt += "Correct answer: "

    return prompt, correct_option


In [4]:
import random

def create_structured_prompt_random(dataset, test_example, num_demonstrations=3):
    # Filter dataset to find samples with the no same context
    no_same_context_samples = [sample for sample in dataset if sample['id'] != test_example['id']]

    if len(no_same_context_samples) <= num_demonstrations:
        raise ValueError("Not enough unique samples with the no same context to select the required number of demonstrations.")

    demonstrations = random.sample(no_same_context_samples, num_demonstrations)

    f = "Give me an answer for the following question by selecting one of the 3 possible answers 1, 2, 3 based on the provided context.\n"

    prompt = ''
    for demo in demonstrations:
        prompt += f
        prompt += f"Context: \"{demo['context']}\"\n"
        prompt += f"Question: \"{demo['question']}\"\n"

        remaining_samples = [s for s in no_same_context_samples if s != demo]
        possible_answers = random.sample(remaining_samples, 2)
        possible_answers.append(demo)

        random.shuffle(possible_answers)

        options = ['1', '2', '3']
        correct_option = None
        prompt += "Possible Answers:\n"
        for idx, sample in enumerate(possible_answers):
            answer_text = sample['answers']['text'][0]
            option = options[idx]
            prompt += f"{option}: {answer_text}\n"
            if sample == demo:
                correct_option = option

        prompt += f"Correct answer: {correct_option}\n"
        prompt += f"Reasoning: \"{demo['reasoning']}\"\n"

    prompt += f
    prompt += f"Context: \"{test_example['context']}\"\n"
    prompt += f"Question: \"{test_example['question']}\"\n"

    remaining_samples = [s for s in no_same_context_samples if s != test_example]
    if len(remaining_samples) >= 2:
        possible_answers = random.sample(remaining_samples, 2)
    else:
        possible_answers = remaining_samples[:2]  # Handle case where not enough samples are available
    possible_answers.append(test_example)

    random.shuffle(possible_answers)

    options = ['1', '2', '3']
    correct_option = None
    prompt += "Possible Answers:\n"
    for idx, sample in enumerate(possible_answers):
        answer_text = sample['answers']['text'][0]
        option = options[idx]
        prompt += f"{option}: {answer_text}\n"
        if sample == test_example:
                correct_option = option

    prompt += "Correct answer: "

    return prompt, correct_option


In [5]:
# Example usage:

# Assume test_example is given (for illustration purposes, we take the first example from the validation set)
test_example = dataset[0]

# Create a structured prompt
try:
    prompt = create_structured_prompt_random(dataset, test_example, num_demonstrations=3)
    # Display the prompt
    print(prompt)
except ValueError as e:
    print(e)

('Give me an answer for the following question by selecting one of the 3 possible answers 1, 2, 3 based on the provided context.\nContext: "The annual NFL Experience was held at the Moscone Center in San Francisco. In addition, "Super Bowl City" opened on January 30 at Justin Herman Plaza on The Embarcadero, featuring games and activities that will highlight the Bay Area\'s technology, culinary creations, and cultural diversity. More than 1 million people are expected to attend the festivities in San Francisco during Super Bowl Week. San Francisco mayor Ed Lee said of the highly visible homeless presence in this area "they are going to have to leave". San Francisco city supervisor Jane Kim unsuccessfully lobbied for the NFL to reimburse San Francisco for city services in the amount of $5 million."\nQuestion: "Who said the homeless in the area would have to leave?"\nPossible Answers:\n1: mayor Ed Lee\n2: 17\n3: Kawann Short\nCorrect answer: 1\nReasoning: "Mayor Ed Lee is quoted in the c

In [ ]:
print(create_structured_prompt(dataset,dataset[1],3))

('Give me an answer for the following question by selecting one of the 3 possible answers 1, 2, 3 based on the provided context.\nContext: "Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50."\nQuestion: "What team was the NFC champion?"\nPossible Answers:\n1: American Footb

In [6]:
import torch
from transformers import LlamaTokenizer, LlamaForCausalLM

model_path = 'openlm-research/open_llama_3b'
# model_path = 'openlm-research/open_llama_7b'

tokenizer = LlamaTokenizer.from_pretrained(model_path)
model = LlamaForCausalLM.from_pretrained(
    model_path, torch_dtype=torch.float16
)
model.to('cuda')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/534k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 3200, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (k_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (v_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (o_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3200, out_features=8640, bias=False)
          (up_proj): Linear(in_features=3200, out_features=8640, bias=False)
          (down_proj): Linear(in_features=8640, out_features=3200, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )


In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import torch
from collections import defaultdict

def evaluate_model(dataset, num_examples=100,num_dem = 1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # We'll assume dataset is structured with 'validation' split containing the required fields
    #validation_set = dataset['validation']

    # Randomly select a subset of the validation dataset for evaluation
    #test_examples = random.sample(dataset['validation'], 100)
    test_examples = list(dataset)

    correct_answers = 0
    total = 0
    invalid_answers = 0

    for test_example in test_examples:
        try:
            # Generate the structured prompt for the model
            prompt, correct_option = create_structured_prompt(dataset, test_example,num_dem)

            # Encode the prompt to input_ids that can be used by the model
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(device)

            outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=1,
            temperature=1,
            num_return_sequences=1,
            do_sample=True
            )

            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            lines = generated_text.splitlines()[-3:]

            last_line = ""
            for line_index, line in enumerate(lines):
              if line.startswith("Correct answer:"):
                last_line = line[15:].strip()
                if not last_line and line_index + 1 < len(lines):
                  next_line = lines[line_index + 1].strip()
                  if next_line in ['1','2','3']:
                      last_line = next_line
                break

            # Check if predicted token is a valid option (a, b, or c)
            if last_line in ['1', '2', '3']:
                #correct_option = prompt.split("Which is the correct answer?")[-1].strip().split('\n')[0]
                if last_line == correct_option:
                    correct_answers += 1
            else:
              invalid_answers +=1

            total += 1

        except Exception as e:
          continue


    accuracy = correct_answers / total if total > 0 else 0
    invalid_per = (invalid_answers/total)*100 if total > 0 else 0
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Invalid answers percentage: {invalid_per:.2f}%")

    return accuracy,invalid_per

# To use the evaluate_model function, ensure you have the appropriate dataset loaded with the required fields
# dataset = load_dataset('some_dataset_name_here')
# evaluate_model(dataset)









In [18]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import torch
from collections import defaultdict

def evaluate_model_random(dataset, num_examples=100,num_dem = 1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # We'll assume dataset is structured with 'validation' split containing the required fields
    #validation_set = dataset['validation']


    # Randomly select a subset of the validation dataset for evaluation
    #test_examples = random.sample(dataset['validation'], 100)
    test_examples = list(dataset)

    correct_answers = 0
    total = 0
    invalid_answers = 0

    for test_example in test_examples:
        try:
            # Generate the structured prompt for the model
            prompt, correct_option = create_structured_prompt_random(dataset, test_example,num_dem)

            # Encode the prompt to input_ids that can be used by the model
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(device)

            outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=2,
            temperature=1,
            num_return_sequences=1,
            do_sample=True
            )
            #print(prompt)
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            lines = generated_text.splitlines()[-3:]

            last_line = ""
            for line_index, line in enumerate(lines):
              if line.startswith("Correct answer:"):
                last_line = line[15:].strip()
                if not last_line and line_index + 1 < len(lines):
                  next_line = lines[line_index + 1].strip()
                  if next_line in ['1','2','3']:
                      last_line = next_line
                break
            #print(generated_text)
            #print("\n")
            # Check if predicted token is a valid option (a, b, or c)
            if last_line in ['1', '2', '3']:
                #correct_option = prompt.split("Which is the correct answer?")[-1].strip().split('\n')[0]
                if last_line == correct_option:
                    correct_answers += 1
            else:
              invalid_answers +=1

            total += 1

        except Exception as e:
          continue


    accuracy = correct_answers / total if total > 0 else 0
    invalid_per = (invalid_answers/total)*100 if total > 0 else 0
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Invalid answers percentage: {invalid_per:.2f}%")

    return accuracy,invalid_per

# To use the evaluate_model function, ensure you have the appropriate dataset loaded with the required fields
# dataset = load_dataset('some_dataset_name_here')
# evaluate_model(dataset)









Random context:

In [19]:
evaluate_model_random(dataset,1000,1)

Accuracy: 0.29
Invalid answers percentage: 20.17%


(0.2866666666666667, 20.166666666666664)

In [20]:
evaluate_model_random(dataset,1000,3)

Accuracy: 0.29
Invalid answers percentage: 6.50%


(0.2866666666666667, 6.5)